In [ ]:
from pathlib import Path
import gc
import pickle

import numpy as np
import pandas as pd
import torch
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion

cwd = Path.cwd().resolve()

def find_release_root(start: Path) -> Path:
    sentinel_dirs = ('Step0_Data', 'Step7_ModelTraining')
    nested_release_dirs = ('Global-solid-waste-prediction',)
    for candidate in [start, *start.parents]:
        if all((candidate / name).is_dir() for name in sentinel_dirs):
            return candidate
        for release_dir in nested_release_dirs:
            release_candidate = candidate / release_dir
            if all((release_candidate / name).is_dir() for name in sentinel_dirs):
                return release_candidate
    raise FileNotFoundError('Could not locate GitHub release root from the current working directory')

release_root = find_release_root(cwd)
output_dir = release_root / 'Step7_ModelTraining' / '2_Results'
WASTE_TYPES = ['AW', 'CDW', 'IW', 'MSW']
TABPFN_N_ESTIMATORS = 8
TABPFN_SUBSAMPLE_SAMPLES = 2000
SEED = 42
BATCH_SIZE = 512
PREDICT_PROGRESS_EVERY = 2
CONTEXT_COLUMNS = [
    'Country Code',
    'Country Name',
    'Year',
    'IncomeGroup',
    'Region',
    'WasteFlag',
]
RAW_FEATURE_COLUMNS = pd.read_csv(output_dir / '0_feature_contract_raw.csv')['feature'].astype(str).tolist()
PREDICTION_LONG_COLUMNS = [
    *CONTEXT_COLUMNS,
    *RAW_FEATURE_COLUMNS,
    'Label_Raw_Observed',
    'Label_Log_Observed',
    'Predicted_Raw',
    'Predicted_Log',
    'Final_Raw',
    'Final_Log',
    'Source',
    'HasObservedLabel',
]

In [ ]:
def resolve_model_version():
    if hasattr(ModelVersion, 'V2_6'):
        return getattr(ModelVersion, 'V2_6')
    if hasattr(ModelVersion, 'V2_5'):
        return ModelVersion.V2_5
    return ModelVersion.V2

def predict_in_batches(model, features, batch_size=4096, progress_every=4):
    n_rows = len(features)
    if n_rows == 0:
        return np.asarray([], dtype=float)
    if batch_size is None or batch_size <= 0 or batch_size >= n_rows:
        return np.asarray(model.predict(features), dtype=float).reshape(-1)
    predictions = []
    n_batches = (n_rows - 1) // batch_size + 1
    for batch_idx in range(n_batches):
        start = batch_idx * batch_size
        end = min((batch_idx + 1) * batch_size, n_rows)
        batch_features = features.iloc[start:end] if hasattr(features, 'iloc') else features[start:end]
        batch_pred = np.asarray(model.predict(batch_features), dtype=float).reshape(-1)
        predictions.append(batch_pred)
        if progress_every and (((batch_idx + 1) % progress_every == 0) or (batch_idx + 1 == n_batches)):
            print(f'Prediction batch {batch_idx + 1}/{n_batches} ({end}/{n_rows} rows)')
    return np.concatenate(predictions, axis=0)

def compute_metrics_frame(prediction_meta, predictions_log):
    has_observed_series = pd.Series(prediction_meta['HasObservedLabel'], copy=False)
    obs_mask = has_observed_series.fillna(False).astype(bool).to_numpy()
    y_true_log = np.asarray(pd.to_numeric(prediction_meta.loc[obs_mask, 'Label_Log'], errors='coerce'), dtype=float)
    y_pred_log = np.asarray(predictions_log, dtype=float)[obs_mask]
    valid_mask = ~np.isnan(y_true_log)
    y_true_log = y_true_log[valid_mask]
    y_pred_log = y_pred_log[valid_mask]
    y_true_orig = np.expm1(y_true_log)
    y_pred_orig = np.expm1(y_pred_log)
    eps = 1e-12
    wape_micro = np.sum(np.abs(y_true_log - y_pred_log)) / max(np.sum(np.abs(y_true_log)), eps)
    ss_res_log = np.sum((y_true_log - y_pred_log) ** 2)
    ss_tot_log = np.sum((y_true_log - np.mean(y_true_log)) ** 2)
    r2_log = 1.0 - ss_res_log / max(ss_tot_log, eps)
    ss_res_orig = np.sum((y_true_orig - y_pred_orig) ** 2)
    ss_tot_orig = np.sum((y_true_orig - np.mean(y_true_orig)) ** 2)
    r2_orig = 1.0 - ss_res_orig / max(ss_tot_orig, eps)
    mae_orig = float(np.mean(np.abs(y_true_orig - y_pred_orig)))
    return pd.DataFrame([{
        'WAPE_micro': float(wape_micro),
        'R2_log': float(r2_log),
        'WAPE_macro': float(wape_micro),
        'R2_original': float(r2_orig),
        'MAE_original': mae_orig,
        'rows': int(valid_mask.sum()),
    }])

In [ ]:
model_version = resolve_model_version()
metrics_frames = []
prediction_parts = []
model_version

In [ ]:
for waste_type in WASTE_TYPES:
    train_processed = pd.read_csv(output_dir / f'0_training_processed_{waste_type}.csv')
    prediction_processed = pd.read_csv(output_dir / f'0_prediction_processed_{waste_type}.csv')
    prediction_raw = pd.read_csv(output_dir / f'0_prediction_raw_{waste_type}.csv')
    feature_columns = pd.read_csv(output_dir / '0_feature_contract_processed.csv')['feature'].astype(str).tolist()
    y_train = train_processed.pop('Label_Log').to_numpy()
    train_processed = train_processed[feature_columns].copy()
    missing_raw_columns = [column for column in RAW_FEATURE_COLUMNS if column not in prediction_raw.columns]
    if missing_raw_columns:
        raise RuntimeError(f'Missing raw feature columns for {waste_type}: {missing_raw_columns}')
    prediction_meta = prediction_processed[CONTEXT_COLUMNS + [
        'Label_Raw',
        'Label_Log',
        'HasObservedLabel',
        'RowRole',
    ]].copy()
    raw_feature_frame = prediction_raw[CONTEXT_COLUMNS + RAW_FEATURE_COLUMNS].copy()
    raw_feature_frame['__raw_match__'] = True
    prediction_meta = prediction_meta.merge(
        raw_feature_frame,
        on=CONTEXT_COLUMNS,
        how='left',
        validate='one_to_one',
        sort=False,
    )
    matched_all = bool(prediction_meta['__raw_match__'].fillna(False).astype(bool).all())
    if not matched_all:
        raise RuntimeError(f'Failed to align raw prediction features for {waste_type}')
    prediction_meta = prediction_meta.drop(columns='__raw_match__')
    x_all = prediction_processed[feature_columns].copy()
    model = TabPFNRegressor.create_default_for_version(
        model_version,
        n_estimators=TABPFN_N_ESTIMATORS,
        device='cuda' if torch.cuda.is_available() else 'cpu',
        fit_mode='low_memory',
        memory_saving_mode='auto',
        ignore_pretraining_limits=True,
        inference_config={'SUBSAMPLE_SAMPLES': TABPFN_SUBSAMPLE_SAMPLES},
        random_state=SEED,
    )
    print(f'Training {waste_type} finalize model on {len(train_processed)} rows with {model_version.value}...')
    model.fit(train_processed, y_train)
    predictions = predict_in_batches(model, x_all, batch_size=BATCH_SIZE, progress_every=PREDICT_PROGRESS_EVERY)
    with open(output_dir / f'1_finalize_model_{waste_type}.pkl', 'wb') as handle:
        pickle.dump(model, handle)
    pred_raw = np.expm1(predictions)
    observed_raw = np.asarray(pd.to_numeric(prediction_meta['Label_Raw'], errors='coerce'), dtype=float)
    observed_log = np.asarray(pd.to_numeric(prediction_meta['Label_Log'], errors='coerce'), dtype=float)
    has_observed = pd.Series(prediction_meta['HasObservedLabel'], copy=False).fillna(False).astype(bool).to_numpy()
    final_raw = np.where(has_observed, observed_raw, pred_raw)
    final_log = np.where(has_observed, observed_log, predictions)
    source = np.where(has_observed, 'observed', 'predicted')
    prediction_part = prediction_meta[CONTEXT_COLUMNS + RAW_FEATURE_COLUMNS].copy()
    prediction_part['Label_Raw_Observed'] = observed_raw
    prediction_part['Label_Log_Observed'] = observed_log
    prediction_part['Predicted_Raw'] = pred_raw
    prediction_part['Predicted_Log'] = predictions
    prediction_part['Final_Raw'] = final_raw
    prediction_part['Final_Log'] = final_log
    prediction_part['Source'] = source
    prediction_part['HasObservedLabel'] = has_observed
    prediction_parts.append(prediction_part)
    metrics = compute_metrics_frame(prediction_meta, predictions)
    metrics.insert(0, 'WasteFlag', waste_type)
    metrics_frames.append(metrics)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
predictions_all = pd.concat(prediction_parts, ignore_index=True)
predictions_all = predictions_all[PREDICTION_LONG_COLUMNS]
predictions_all = predictions_all.set_index(['Country Code', 'Year', 'WasteFlag']).sort_index().reset_index()
predictions_all = predictions_all[PREDICTION_LONG_COLUMNS]
predictions_all.to_csv(output_dir / '1_predictions_all_wastes_long.csv', index=False)

metrics_all = pd.concat(metrics_frames, ignore_index=True)
metrics_all.to_csv(output_dir / '1_prediction_metrics_by_waste.csv', index=False)
metrics_all